# Meetings - Notes & Questions

## Modeling interest rates

notes ...

## Deep hedging

Fait:
- Deep hedging vs Black-Scholes avec et sans transaction costs


En cours:
- Objectif de faire comme dans le papier sur RL hedging options, mais juste avec le RL et des simulations differente (pas encore d'option), faire les memes graphique et analyser
- Simuler Heston, GARCH, Jump-diffusion process, afin de voir la difference entre deep hedging et black-scholes hedging
- Train deep hedging agent sur plusieurs model (batch-style)

Question:
- utiliser Black-Scholes delta comme base sur d'autre model (ex. Heston)
- Overview des references vers Francois et Al. pour simuler des options prices
- Comment fonctionne delta-gamma hedging (hedge avec un option supplementaire, transaction costs?)


a apprendre:

Autoencoder
NN generatif


## SANOS: Smooth strictly Arbitrage-free Non-parametric Option Surfaces

Summary of Buehler, Horvath, Kratsios, Limmer, Saqur (2026) - `SANOS - Option surface.pdf`

**Core idea**: A method to build option price surfaces $\hat C(T,K)$ that are simultaneously *smooth* and *strictly arbitrage-free* - something existing approaches (SSVI, plain linear interpolation, heavy stochastic-vol calibration) don't achieve together. It's framed as a smooth generalization of the well-known linear interpolation scheme for arbitrage-free option prices.

**Key construction** (eq. 1):
$$\hat C(T_j, K) = \sum_i q_j^i \,\text{Call}(K_j^i, K, V_j)$$
Call prices are convex combinations of Black-Scholes call payoffs anchored at model strikes $K_j^i$, with weights $q_j^i$ that behave like a discrete martingale transition density (must be non-negative, sum to 1, and satisfy the martingale property $K_j \cdot q_j = K_{j-1} \cdot q_{j-1}$-style constraints). Setting the smoothness parameter $\eta=0$ recovers plain linear interpolation (non-smooth); $\eta \to 1$ over-smooths to almost only fitting ATM. $\eta=0.25$ is their recommended default.

**Why it matters**: Linear interpolation is arbitrage-free but is proven (citing Buehler 2006) to be "the most expensive" interpolation - it systematically overprices between quoted strikes, which shows up as implied vol bumps between strikes (Figure 2). SANOS fixes that while keeping strict no-arbitrage.

**Theoretical backbone**: Theorem 2.2 gives 5 shape conditions (unit expectation, no atom at 0, decay to 0, convexity in K, monotonicity in T) that are necessary and sufficient for a call surface to correspond to an actual positive martingale (Fundamental Theorem of Asset Pricing). Theorem 3.1 shows that replacing the discrete jump anchor with call prices under a smooth martingale $Y$ (they use log-normal, i.e. Black-Scholes) preserves all 5 conditions, hence "smooth + arbitrage-free."

**Calibration**: Fitting $q$ to market bid/ask is a **linear program** (or convex program with bid/ask penalties) - this is the big practical win: no nonlinear optimization, sub-second fits to full SPX surfaces across 48 expiries (91.4% of options fit within bid/ask).

**Extra contribution**: An equivalent parametrization via "discrete local volatilities" (Sigma, extending Buehler & Ryskin 2015) where the *only* constraint is positivity - useful for generative/ML models of option surfaces, since you can just exponentiate or square unconstrained NN outputs to get valid parameters.

**Relevant to project**: Could feed directly into building a smooth vol surface as an input/testbed for Deep Hedging plots (see `Ploting_DH.py`).

$$\binom{N+K-1}{K} = \frac{(N+K-1)!}{K!,(N-1)!}$$

In [3]:
import numpy as np
x = [1,2,1,1,1,2,1,3]
print(np.cumsum(x))

[ 1  3  4  5  6  8  9 12]


In [4]:
print(6%2)

0


**Overview**
- Model-free method to fit arbitrage-free implied vol surfaces by casting calibration as an (approximately) convex QP — bridges "flow" desk needs (intuitive params, speed) and "exotics" desk needs (arb-free everywhere incl. tails, needed for SLV calibration)
- Parameterizes **variance** $v=\sigma^2$ (not price, not vol) as a function of normalized log-moneyness $z = \frac{\log(K/F)}{\sigma_\star\sqrt{T}}$, via a **dual representation**:
  - Cubic spline space (intuitive/physical): ATM variance $v(0)$, ATM skew $\partial v/\partial z|_0$, and per-knot convexities $\partial^2v/\partial z^2|_{z_i}$
  - B-spline space (mathematical): linear basis-function weights
  - The two are linked by a fixed **linear transformation** — this duality is the central trick that keeps an intuitively-parameterized, arbitrarily flexible surface inside a convex/linear optimization
- Any number of knots per expiry (no restriction), one regularization hyperparameter $\lambda$ works across all underlyings with no tuning — contrasts with SVI (5 params, notoriously delicate non-convex calibration even with Zeliade's 2-step trick)
- All expiries calibrated jointly in a single optimization (not sequential expiry-by-expiry), so no-calendar-arbitrage constraints link them directly

**Objective function** (terms scaled to behave like a chi-squared, so one $\lambda$ works everywhere)
- Least-squares to mid, weighted by inverse squared bid-ask spread in variance space → wide/illiquid quotes barely move the fit
- "Above-ask" / "below-bid" penalties (vega-weighted) for quotes that only have one side (e.g. far OTM options that are offer-only) — without these the fit can wander outside the bid-ask in the tails
- Strike regularization: penalizes total variation of the normalized convexity $c$ across knots → smooths the smile, trades off against fit quality via $\lambda$ (e.g. $\lambda=0.05$ empirically works everywhere)

**Arbitrage-free constraints** (linear in variance space on a fixed knot/strike grid)
- No-calendar-arbitrage: total variance $vT$ increasing in $T$ at fixed $K/F$ — exactly linear, no approximation needed
- No-butterfly-arbitrage (PDF $\geq 0$ via Breeden–Litzenberger): nonlinear/nonconvex in variance space in general. **Main technical contribution of the paper**: derive an explicit linearization of this constraint around the previous solution, so it's enforced via a couple of QP passes (iteration 1 without the constraint, iteration 2+ with it linearized at iteration 1's solution) — 2 iterations usually suffice
- Lee's moment-formula tail slope bounds enforced at the edge knots (variance is linearly extrapolated beyond them)
- Simple positivity of variance

**Solver**
- Canonical QP $\min \tfrac12 x^TPx+q^Tx$ s.t. $l\le Ax\le u$, built via CVXPY, solved with Clarabel (Oxford's interior-point solver — found faster & more robust than OSQP for this problem)
- Example: SPX (14,500 vol quotes, 46 expiries, 20 params/expiry) calibrates in ~0.15s for the full 2-iteration fit; calibration time scales roughly linearly in # params in the typical regime (quadratic only once param counts get very large) — sharp contrast to nonconvex smile fitting, which tends to degrade badly as dimensionality grows

**Notable**: because there's no smile-shape assumption baked into the parameterization, CVI can fit W-shaped/"moustache" smiles (e.g. pre-earnings NDX — negative ATM convexity → bimodal implied PDF) that classical parametric models (SVI, SSVI) can't represent, while remaining provably arb-free.

The actual trade this rules out: a calendar spread. Sell a call at strike $K$ expiring $T_1$, buy a call at the same relative strike (same $K/F$) expiring $T_2 > T_1$, and collect the net premium today if the near-dated call is priced richer than the far-dated one. If that's ever profitable with certainty, it's arbitrage — so no-arbitrage forces the far-dated (normalized) call to always be worth at least as much as the near-dated one.

Why it must hold — a completely model-free argument. Write the normalized underlying $Z_t = S_t/F_t$, a martingale. A call payoff $(z-K)^+$ is convex. For $T_1 < T_2$, conditional Jensen plus the martingale property gives:
$$E\big[(Z_{T_2}-K)^+ \mid \mathcal F_{T_1}\big] ;\geq; \big(E[Z_{T_2}\mid \mathcal F_{T_1}] - K\big)^+ = (Z_{T_1}-K)^+.$$
The right side is exactly what you owe on the near-dated short call at $T_1$. The left side is exactly the fair market value, at $T_1$, of the far-dated call you're still holding. So at $T_1$, your long position is worth at least enough to cover your short obligation — plus you already banked a positive credit today. That's a locked-in, riskless profit if the price inequality ever goes the wrong way. Notice this uses nothing about Black-Scholes, GBM, or any specific dynamics — just convexity of the call payoff and the martingale property, which is why it's a completely model-free no-arbitrage condition (this is SANOS's Theorem 2.2, condition 5, stated exactly this way).

Why fixed $K/F$, not fixed $K$: different expiries generally have different forwards (dividends, rates), so comparing the same nominal strike $K$ across $T_1$ and $T_2$ compares different relative positions in the distribution. The theorem is really about the normalized process $Z_T = S_T/F_T$ at a fixed normalized strike, which translates back to "cash" terms as fixed strike-to-forward ratio.

Why total variance specifically, not vol itself. This part is really just an artifact of the Black-Scholes formula, not a new piece of financial theory. The BS price for a normalized call, $\text{Call}(1,K,\sigma^2T)$, depends on $\sigma$ and $T$ only through the single combination $w := \sigma^2 T$ — you can see this directly in $d_\pm = \dfrac{-\ln K \pm \frac12 w}{\sqrt w}$: there's no $\sigma$ or $T$ appearing separately, only $w$. And the BS price is strictly increasing in $w$ for fixed $K$ (this is just vega being positive — more variance always makes optionality strictly more valuable). So when you take the model-free statement "price must be non-decreasing in $T$" and re-express it in implied-vol coordinates, it necessarily becomes "the unique quantity price is monotonic in — total variance $w=\sigma^2T$ — must be non-decreasing in $T$." It's not that total variance is special financially; it's that it's exactly the coordinate BS price is monotonic in, so the price condition transfers to it term-for-term.

The counterintuitive payoff of this: implied vol itself is allowed to decrease with maturity — vol term structures are routinely downward-sloping or humped (e.g. an elevated near-dated vol from an upcoming earnings print, decaying into a calmer longer-term level). What's forbidden is the product $\sigma^2T$ ever decreasing — i.e. $T$ growing can (and usually does) more than compensate for $\sigma$ falling. So "no calendar arbitrage" is a much weaker, more permissive condition than "vol must be increasing in $T$" — it only bites when a term-structure kink is steep enough that accumulated variance actually shrinks, which is the genuinely pathological case it's designed to catch.

It's the fact that the same variance curve $v(z)$ can be written in two different coordinate systems, related by a fixed linear map — and it's directly relevant to what you're building right now in Stage 1.

Representation 1 — B-spline weights (the "mathematical" space, what you're constructing now):
$$v(z) = \sum_{i=1}^{n+2} \alpha_i B_i(z)$$
The $\alpha_i$ are just weights on basis bumps. They have no financial meaning on their own — $\alpha_3$ being large doesn't tell you anything about skew or convexity directly.

Representation 2 — cubic-spline / "physical" parameters (the intuitive space):
the same function $v(z)$, described instead by $n+2$ specific quantities:

$v(0)$ — the ATM variance
$\partial v/\partial z\big|_{z=0}$ — the ATM skew
$\partial^2 v/\partial z^2\big|_{z_i}$ for each of the $n$ knots — the convexities at each knot
These are exactly the numbers a trader would ask for.

Why "dual": both describe the same space of functions (piecewise-cubic on a fixed knot grid), and there's a fixed linear transformation between the two parameter vectors — a $(n{+}2)\times(n{+}2)$ matrix computed once from the knot locations, not from the market data. Since it's linear, you can convert either direction (matrix multiply one way, solve a linear system the other way) without losing anything or introducing any nonlinearity.

Where this transformation actually comes from, concretely — and this connects straight to your Stage 1 code: once you have the basis functions $B_i(z)$ and their derivatives, the physical parameters are just specific linear functionals of the weight vector $\alpha$:
$$v(0) = \sum_i \alpha_i B_i(0), \qquad s = \sum_i \alpha_i B_i'(0), \qquad c_j = \sum_i \alpha_i B_i''(z_j).$$
So if you stack $B_i(0)$, $B_i'(0)$, and $B_i''(z_j)$ for every $i,j$ into a matrix $T$, then (physical params) $= T\alpha$ — that matrix $T$ is the dual transformation. You'll build it in Stage 2 by evaluating your basis functions and their first/second derivatives at $z=0$ and at each knot $z_j$; scipy's BSpline objects support .derivative(nu=1) / .derivative(nu=2) to get derivative splines directly, which will make constructing those rows straightforward.

Why bother with two representations instead of just picking one: they're each good at a different job.

The B-spline weights are what you actually hand to the optimizer — they enter $v(z)$ linearly, which keeps the QP objective quadratic, and their local support keeps the constraint/objective matrices sparse (this sparsity is exactly why Clarabel solves the problem in hundredths of a second even with thousands of parameters).
The physical parameters are what you read, plot, and constrain — the paper's derived no-arbitrage formulas (Lee's tail bounds, the linearized butterfly condition) are written directly in terms of $s$ and $c$, and they're what you'd report to a trader or feed into a factor.
Practically: you never have to choose one over the other. Since $T$ is a fixed linear map, you can express the entire CVI objective and every constraint purely in terms of $\alpha$ (substituting $v(0)=T_1\alpha$, $s=T_2\alpha$, etc. wherever the physical parameters appear), and CVXPY never even needs to see the physical parameters as separate variables — they're just linear readouts of $\alpha$ for you to inspect after solving.

n those plots (Figures 6, 7, 8, 9, 12, 13, 14 — the blue-shaded curve on the right axis), the "pdf" is the risk-neutral probability density of the underlying's terminal price, implied by the fitted volatility curve via the Breeden–Litzenberger formula: $\text{pdf}(K) = e^{rT},\partial^2C/\partial K^2$. It's not an input or an assumption — it's a derived diagnostic, computed after the fact from whatever smile CVI just produced.

Where the formula comes from (this is Appendix A.1, Equation 9 in the paper): differentiating the Black-Scholes price twice with respect to strike, but treating $\sigma$ as a function of $K$ (the fitted smile) rather than a constant, gives:

$$\text{pdf}(K) = \phi(d_2)\left(\frac{1}{K\sigma\sqrt T} + 2\frac{d_1}{\sigma}\frac{\partial\sigma}{\partial K} + \frac{d_1d_2K\sqrt T}{\sigma}\Big(\frac{\partial\sigma}{\partial K}\Big)^2 + K\sqrt T,\frac{\partial^2\sigma}{\partial K^2}\right)$$

where $\phi$ is the standard normal density. The first term alone, $\phi(d_2)/(K\sigma\sqrt T)$, would be the density if the smile were flat (i.e. plain lognormal/Black-Scholes) — but the other three terms are corrections driven by the slope and curvature of the smile itself ($\partial\sigma/\partial K$, $\partial^2\sigma/\partial K^2$). This is exactly why a smile with steep skew or strong convexity produces a non-lognormal implied distribution (fat tails, skewed mass, or in the W-shaped/moustache case, bimodal) — the market isn't pricing a lognormal underlying, and this formula extracts what it is actually pricing.

Why it's plotted at all — it's the visual arbitrage check. No-butterfly-arbitrage is by definition "this density is non-negative everywhere." So plotting it is literally showing you whether the fit is arbitrage-free at a glance, not just reporting a nice statistic. The paper uses this explicitly: Figure 13 shows the same AAPL expiry fit twice — (a) without the linearized butterfly constraint, where the pdf dips visibly negative (shaded pink) around two strike regions because the fitted skew is too steep / smile too concave there, and (b) with the constraint applied, where the fit is pushed away from the raw mid-price just enough to keep the density non-negative everywhere, while staying within the wider bid-ask bounds.

So concretely: whenever you see that shaded density curve peak near the forward and taper off in the wings, that shape is a consequence of the fitted smile's slope and curvature at each strike — it's the same information as the vol smile, just transformed into "what does the market think the terminal price distribution looks like," and its non-negativity is the entire mathematical content of "no butterfly arbitrage."

The Lebesgue measure is the rigorous mathematical generalization of "length" (on $\mathbb R$), "area" (on $\mathbb R^2$), "volume" (on $\mathbb R^n$) to a vastly larger class of sets than just intervals/rectangles — while still agreeing with ordinary length on the sets you already know how to measure: $\lambda([a,b]) = b-a$.

Why you need more than "length of an interval." Elementary length only handles finite unions of intervals cleanly. The moment you want to integrate more exotic functions, or take limits of integrals (does $\int \lim_n f_n = \lim_n \int f_n$?), the naive Riemann-integration theory breaks down in ways that matter for real analysis and probability. Lebesgue's construction fixes this, and the payoff is clean convergence theorems (Monotone/Dominated Convergence) that are indispensable once you're doing anything with limits of random variables or integrals — which is constantly, in probability theory.

Roughly how it's built: for any set $A\subseteq\mathbb R$, define an outer measure by covering $A$ with countably many intervals and taking the infimum of total interval length over all such covers:
$$\lambda^(A) = \inf(\sum_k \text{length}(I_k) : A\subseteq \bigcup_k I_k)$$
Then restrict attention to sets $E$ that "split" every other set additively (the Carathéodory criterion): $\lambda^(A) = \lambda^(A\cap E) + \lambda^(A\setminus E)$ for all $A$. Those $E$ are the Lebesgue measurable sets, and $\lambda^*$ restricted to them — now called $\lambda$ — is countably additive: the measure of a countable disjoint union equals the sum of the measures. That's the Lebesgue measure. (Almost every set you'll ever write down is measurable; you need the Axiom of Choice to even construct a non-measurable one.)

Key properties: countably additive, translation-invariant ($\lambda(A+x)=\lambda(A)$), $\lambda([a,b])=b-a$, and any countable set (e.g. the rationals) has measure exactly zero.

Why this connects directly to what you were just asking about ("is it a density"). The precise, rigorous meaning of "random variable $X$ has a density $f$" is: the probability measure $P_X$ (the law of $X$) is absolutely continuous with respect to Lebesgue measure, and by the Radon–Nikodym theorem there's a function $f = dP_X/d\lambda$ such that $P_X(A) = \int_A f, d\lambda$ for every measurable $A$. Not every distribution has this — a distribution can also have an atom: a single point $x_0$ with $P(X=x_0) > 0$. A single point has Lebesgue measure zero, so a point mass carries probability that no density function (integrated against $\lambda$) can ever capture — it's "singular" with respect to Lebesgue measure, invisible to $f$.

This is exactly the mechanism behind SANOS's condition "zero is unattainable," $\partial_KC(T,0)\equiv -1$ (Remark 2.4 in the appendix). The proof there literally constructs the law of $Z_T$ as $\nu(A) = (1+c'(0)),\delta_0(A) + \tilde\nu(A)$ — a Lebesgue-absolutely-continuous part $\tilde\nu$ (a genuine density, the Breeden–Litzenberger object you were just asking about) plus a possible Dirac point mass $\delta_0$ at zero. The weaker boundary condition $\partial_KC(T,0)\geq -1$ allows that atom to be present (positive probability of the underlying hitting exactly zero); the stricter condition $=-1$ that CVI/SANOS both impose forces the atom's weight to zero, guaranteeing the distribution is purely absolutely continuous — i.e., guaranteeing that the second-derivative formula you computed earlier actually is the whole story, with no hidden spike of probability the formula can't see.

Duality in convex analysis is the idea that every convex optimization problem (or convex function) has a "mirror" representation that encodes the same information, viewed from the dual space.

Core object: the convex conjugate (Fenchel conjugate)

For a function $f: \mathbb{R}^n \to \mathbb{R} \cup {+\infty}$, define

$$ f^*(y) = \sup_x { \langle y, x \rangle - f(x) } $$

$f$ is always convex (even if $f$ isn't), since it's a supremum of affine functions of $y$. Geometrically, $f^(y)$ tells you the maximum "gap" between the linear function $\langle y,\cdot\rangle$ and $f$ — equivalently, it encodes $f$ via its supporting hyperplanes rather than its epigraph directly.

If $f$ is convex, proper, and lower semicontinuous, then $f^{**} = f$: you recover $f$ exactly from its conjugate. This is the Fenchel-Moreau theorem, and it's the reason duality is "lossless" for convex functions (it can fail for nonconvex ones — you only ever recover the convex hull).

Lagrangian duality

For a primal problem
$$\min_x f_0(x) \text{ s.t. } f_i(x) \le 0$$
the Lagrangian $L(x,\lambda) = f_0(x) + \sum \lambda_i f_i(x)$ gives a dual function
$$g(\lambda) = \inf_x L(x,\lambda)$$
which is concave regardless of convexity of the primal (it's an infimum of affine functions of $\lambda$). Maximizing $g(\lambda)$ over $\lambda \ge 0$ is the dual problem.

Weak duality always holds: $g(\lambda) \le f_0(x)$ for any feasible $x, \lambda$ — the dual is a lower bound on the primal.
Strong duality (dual optimal = primal optimal) holds under constraint qualifications like Slater's condition when the primal is convex.
Fenchel duality connects the two views: for $\min_x f(x) + g(Ax)$, the dual is $\max_y -f^(A^Ty) - g^(-y)$, with the conjugate playing the role Lagrangian relaxation plays in the constrained case. KKT conditions are the first-order optimality glue linking primal and dual solutions.

Intuition you may find familiar from finance: the conjugate/duality machinery here is exactly what underlies utility-indifference pricing and the dual formulation of expected utility maximization (Kramkov-Schachermayer style), where $U^*$ (the conjugate of a utility function) turns a primal terminal-wealth optimization into a dual problem over equivalent martingale measures / state-price densities. Same Fenchel conjugate, same $f^{**}=f$ biconjugation trick.

Want me to go deeper on any piece — e.g., the geometric picture (supporting hyperplanes / epigraphs), a worked example (conjugate of $|x|$, or of exponential utility), or the finance-duality connection specifically?

SANOS — Summary
Core idea: represent the call price surface directly as a convex combination ("mixture") of Black–Scholes call payoffs anchored at market strikes, rather than fitting a smile shape and then checking/imposing arbitrage-freeness:
$$\hat C(T_j,K) = \sum_i q_j^i,\text{Call}(K_j^i, K, V_j)$$
The weights $q_j^i \geq 0$ (summing to 1, unit-mean in strike) are exactly a discrete martingale/transition density. Because a single BS call price is itself always convex and monotone in strike for fixed variance, any convex combination of them is automatically convex and monotone too — arbitrage-freedom is structural, not an optimization constraint you have to derive or linearize (unlike CVI's variance-space butterfly condition).

Relation to linear interpolation: SANOS is explicitly framed as a "smooth generalization of linear interpolation." Plain linear interpolation of market prices is already arbitrage-free (Prop 2.13/2.14) but not smooth, and — per Buehler's earlier "expensive martingales" result — it's provably the most expensive arbitrage-free interpolant (implied vol between quoted strikes ends up systematically too high). Setting the BS-kernel variance to $\eta V_j$ and increasing $\eta$ from 0 smooths this out: $\eta=0$ reduces exactly to linear interpolation, $\eta\to1$ can only fit the ATM point. $\eta=0.25$ is the recommended default.

Calibration is a Linear Program, not a QP: since model prices are linear in $q$, fitting to market bid/ask (with weights, and penalties for one-sided quotes) is an LP — materially cheaper than most IV-surface fitting routines. Multi-expiry consistency (no calendar arbitrage) comes from chaining martingale transition operators $Q_{j|j-1}$ between expiries, so it's enforced by construction too, not as a separate inequality.

Accessible/generative parametrization (the piece your research notes already flagged as key): the martingale density $q$ can be re-expressed via Discrete Local Volatilities $\Sigma_j^i \geq 0$ (from Buehler–Ryskin — the paper we're about to read) — the only constraint is non-negativity. Transition operators are built by inverting a tridiagonal ("tri-band") matrix whose structure (Z-matrix/M-matrix) guarantees the inverse is automatically a valid probability transition matrix — cheap ($O(N)$) and provably valid. This is what makes SANOS suitable as an output layer for generative models: any positivity-constrained decoder output (e.g. softplus) becomes a guaranteed arbitrage-free surface with zero extra constraint-checking.

Empirical result: fit to SPX, 1000 options across 48 expiries (1–657 business days) on a single day — 91.4% of options fit within bid/ask, median error on the rest just 21% of half-spread, sub-second on a desktop.

Explicit positioning vs. CVI: SANOS's related-work section cites CVI directly among prior IV-interpolation attempts and argues "none of the proposed models is able to ensure strict absence of arbitrage" — i.e., constraint-based/linearized approaches like CVI are approximate where SANOS's construction is exact. Its own comparison table: linear interpolation (arb-free✓, smooth✗, linear complexity) vs. SSVI (arb-free✓, smooth✓, parametric, high complexity) vs. SANOS (arb-free✓, smooth✓, non-parametric, linear complexity) — claiming to be the first to get all three at once.

Scope/limitations the authors flag: European options only (American needs de-Americanization first); this is a static, single-snapshot construction — dynamic (across-time) arbitrage-freeness of a simulator built on top of it is explicitly left as future work, combined with the "Deep Hedging: Learning to Remove the Drift" machinery — exactly the research direction your own notes already point at.

Now let me read the Discrete Local Volatility paper to work out precisely how it connects.

Read C:\Users\victo\OneDrive\Documents\Summer Project\Courses and lectures\Volatility Modeling\Discrete Local Volatility for Large Time Step.pdf
Direct lineage: SANOS's "accessible/generative parametrization" (its Section 3.2, Discrete Local Volatilities $\Sigma_j^i$) is explicitly built on this paper — SANOS says outright "we recall results from [BR15]... to fix notation and ideas," and "This material is from [BR15] with notations aligned." Here's exactly what gets carried over, and what doesn't.

1. The core trick SANOS reuses: tri-band matrix → guaranteed-valid transition operator.
This paper's Theorem 3.3 ("Construction of Transition Matrices") is the linear-algebra fact both papers hinge on: build a matrix $M$ that's a Z-matrix (positive diagonal, non-positive off-diagonal) with columns summing to 1 — a "non-singular M-matrix" — and its inverse $M^{-1}$ is automatically non-negative with columns summing to 1, i.e. a valid transition/probability matrix, no further checking required. This paper applies it via the tri-band $I_j^{-1}$ (Eq. 36) built from Backward Local Volatility $\varsigma_j^i$. SANOS's Theorem A.4 is essentially the same proof, word-for-word in spirit, applied to its own tri-band $Q_{j|j-1}^{-1}$ built from Discrete Local Volatility $\Sigma_j^i$. Same trick, same source ([AH11], credited in both), repackaged.

2. Why "Backward" specifically — this is the part SANOS silently inherits and that matters most.
This paper draws a sharp distinction between Forward Local Volatility (built from Forward-Theta, the classic Dupire-style discretization) and Backward Local Volatility (built from Backward-Theta). It proves:

Positivity of Backward-Theta, not Forward-Theta, is what's actually equivalent to absence of arbitrage (Theorem 3.2) — there's an explicit counterexample (Section 3.1, Fig. 3.1) where Forward-Theta is positive everywhere but the surface still has arbitrage.
The implicit (backward-based) FD operator is unconditionally a valid transition kernel (Prop 4.2/4.3) — no extra stability constraint needed — whereas the explicit (forward-based) operator only works if you additionally impose Eq. (30), a CFL-type restriction.
This is exactly why SANOS's DLV parametrization gets away with "only positivity of $\Sigma$ needed, nothing else" — that unconditional-validity result is proven here, not re-derived in SANOS. SANOS's $\Sigma_j^i \geq 0$ is this paper's Backward Local Volatility $\varsigma_j^i \geq 0$, just renamed.

3. The inhomogeneous-strike-grid trick also carries over.
This paper's Section 4.3.3 constructs $\Xi_j = \Psi_j\Omega_j$: given a density $p_{j-1}$ on strikes $K_{j-1}$, linearly interpolate its implied call prices onto the next expiry's (different) strike grid $K_j$, then re-derive a density there — all while providably preserving the martingale property and total mass. SANOS's Theorem 3.6 ($L_{j|j-1}$, mapping $q_{j-1}$ to a density $\bar q_{j-1}$ over $K_j$) is structurally the same construction, needed for exactly the same reason: real market strike grids differ expiry to expiry.

4. Where they diverge — and this is the real payoff of reading both.
This paper is fundamentally a two-stage pipeline: (a) repair/fit a possibly-arbitrage-violating discrete price grid to be arbitrage-free via an L¹ linear program with Gamma/Theta constraints (its Section 3), then (b) derive transition kernels from that fixed grid via the tri-band FD-operator construction (its Section 4), which you then use for backward/forward induction and Monte Carlo pricing of path-dependent products (Bermudan/American digitals — its worked examples). SANOS's production model fuses stage (a) away entirely: because $\hat C = \sum_i q_i,\text{Call}(K_i,K,V)$ is a mixture of smooth BS kernels, convexity and monotonicity come for free from the representation — there's no separate "repair" LP needed, and no raw discrete grid with kinks to smooth over. SANOS only reaches for this paper's machinery (Section 3.2, the DLV branch) as an alternative, more ML-friendly parametrization of the same martingale density — not because it needs the arbitrage-repair step, but because DLV coordinates are convenient for a generative model's output layer.

5. What this paper has that SANOS explicitly leaves as future work.
This paper's Section 4.4 ("Small Steps") — decomposing large-step transition operators via eigendecomposition to get consistent intermediate-time transition kernels — and its whole path-dependent-pricing machinery (backward induction, MC via transition kernels, Likelihood Ratio Greeks) go beyond what SANOS attempts. SANOS is explicitly a static, single-snapshot arbitrage-free surface; using it to simulate forward in time consistently (the actual dynamic pricing engine this paper builds) is exactly the gap SANOS's authors flag as future work, tied to the "Deep Hedging: Learning to Remove the Drift" paper — which is precisely the research direction already in your notebook's SANOS research notes.

One more shared thread worth knowing: both papers lean on Buehler's earlier "Expensive Martingales" [Bu06] result (this paper's Theorem 3.5) — that a martingale built from linear interpolation of prices is provably the "most expensive" one consistent with the data (it dominates every other consistent martingale in convex order). This paper cites it to justify why linear interpolation is a fine starting point for its arbitrage-repair step but not what you'd want for actual pricing; SANOS cites the same result to motivate exactly why its smoothing parameter $\eta$ exists — to move away from that provably-overpriced linear-interpolation baseline.

Neither exactly — it's a third thing, and the distinction matters. Let me unpack precisely what's fixed, what's optimized, and where the "in-between" values actually come from.

Two separate strike grids exist in the paper, and they're not necessarily the same.

Market strikes $k_\ell^r$: where you actually have bid/ask quotes.
Model (anchor) strikes $K_j^i$: the points where you place a Black-Scholes call kernel. The paper explicitly allows these to differ from the market strikes — Section 4.2's production model treats "Model Expiries/Model Strikes" as a separate input from "Market Expiries/Market Strikes," and their own implementation adds extra model strikes wherever market strikes are too sparse.
What's actually being weighted is not market prices — it's smooth BS payoff kernels anchored at those model strikes.
$$\hat C(T_j,K) = \sum_i q_j^i,\text{Call}(K_j^i, K, V_j)$$
Each $\text{Call}(K_j^i, \cdot, V_j)$ is a full smooth function of $K$ (a Black-Scholes call price curve, "puffed out" by variance $V_j$, centered near $K_j^i$) — not a single observed number. The weights $q_j^i$ are a discrete density over the anchor strikes, calibrated by solving a linear program so that $\hat C(T_j, \cdot)$ evaluated at the market strikes lands within (or close to) the observed bid/ask there.

So calibration targets market prices, but the free parameters live on the (possibly different) model grid. This is exactly analogous to CVI's basis-function idea (fit basis-function weights so the resulting curve matches market quotes) — except here the "basis functions" are literal Black–Scholes call prices instead of B-splines.

Where does the smooth "in-between" behavior come from?
It's not a separate interpolation step at all — it's automatic. Since $\text{Call}(K_j^i, K, V_j)$ is already a smooth function of $K$ everywhere (not just at the anchor points), $\hat C(T_j,K) = \sum_i q_j^i,\text{Call}(K_j^i,K,V_j)$ is defined at every strike $K$ by the same formula — the value between two market strikes isn't interpolated from neighboring market prices, it's just the same global weighted-sum-of-kernels evaluated at a new point.

The smoothness parameter $\eta$ is exactly what separates this from "weighted market prices." The kernel used is $\text{Call}(K_j^i,K,\eta V_j)$. As $\eta\to0$, the variance collapses to zero and the BS call kernel degenerates into the plain kinked payoff $(K_j^i-K)^+$ — at that limit, if you also set the anchor strikes equal to the market strikes and $q$ equal to the market-implied discrete density, the formula reduces exactly to linear interpolation of market call prices (this is literally stated in Theorem 3.1: "if $q=p$ and $Y\equiv1$, $\hat C$ reduces to linear interpolation"). So your first framing — "weight of existing market call prices" — is the special/degenerate case $\eta=0$. The whole point of SANOS is to not use that degenerate case: for $\eta>0$ (0.25 recommended), each anchor strike's kernel spreads its probability mass smoothly across nearby strikes instead of concentrating it as a kink exactly at $K_j^i$, which is what produces a genuinely smooth (not piecewise-linear) fitted curve while staying provably arbitrage-free.

Practical summary: pick a strike grid (often close to market strikes, sometimes densified), place a Black-Scholes kernel of variance $\eta V_j$ at each, then solve an LP for the mixture weights $q_j^i$ that make the resulting smooth curve pass through the market bid/ask at the market strikes — and the curve's value at any other strike (between quotes, or beyond them) falls out of that same formula for free, with no separate interpolation rule needed.